# Hybrid LSTM-CNN for IoT Intrusion Detection
**Replicating Sinha et al., 2025** — *A High Performance Hybrid LSTM-CNN Secure Architecture for IoT Environments Using Deep Learning*

### All fixes applied vs original notebook
| # | Cell | Issue | Fix |
|---|------|-------|-----|
| 1 | Cell 2 | `KAGGLE_API_TOKEN` not recognised by kaggle library | Changed to `KAGGLE_USERNAME` + `KAGGLE_KEY` |
| 2 | Cell 3 | `LabelEncoder` imported from `sklearn.decomposition` | Moved to `sklearn.preprocessing` |
| 3 | Cell 3 | `SAMPLE_FRAC = 0.01` (1%) — too small to replicate | Set to `1.0` (full dataset) |
| 4 | Cell 5 | `StandardScaler` — paper specifies Min-Max [0,1] | Replaced with `MinMaxScaler` |
| 5 | Cell 5 | PCA fit on full X before split — data leakage | PCA now fit only on X_train |
| 6 | Cell 6 | `IndentationError` in `build_lstm_cnn` | Fixed indentation |
| 7 | Cell 6 | 3 LSTM layers all `return_sequences=True` | 2 LSTM layers; 2nd `return_sequences=False` + Reshape |
| 8 | Cell 6 | Missing `MaxPooling1D` after each Conv | Added per paper Table 5 |
| 9 | Cell 6 | L2 regularization missing | Added `l2(1e-4)` to Dense/LSTM layers |
| 10 | Cell 6 | Dropout 0.2 for LSTM/RNN/BiLSTM — paper says 0.3 | Fixed per paper Table 4 |
| 11 | Cell 6 | All 5 baseline models commented out | Uncommented CNN, RNN, LSTM, BiLSTM, GRU |
| 12 | Cell 7 | `EPOCHS = 5` — paper uses 50 | Set to `50` |
| 13 | Cell 7 | `BATCH_SIZE = 512` — paper uses 128 | Set to `128` |
| 14 | Cell 7 | `confusion_matrix` assumes binary only | Guarded with binary/multiclass check |

---
Hit **Runtime → Run All** to execute.

In [1]:
# CELL 1: Install Dependencies
%pip install -q numpy pandas scikit-learn scipy tensorflow keras matplotlib seaborn imbalanced-learn pyyaml kaggle joblib tqdm

In [ ]:
# ============================================================
# SPEED PATCH CELL — paste this BEFORE Cell 3 in the notebook
# ============================================================
import tensorflow as tf

# --- 1. Mixed precision (biggest single speedup on T4) ---
# T4 GPUs have dedicated Tensor Cores for fp16 computation.
# This halves memory usage AND doubles throughput with no
# meaningful accuracy loss for classification tasks.
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print(f'Compute dtype:  {tf.keras.mixed_precision.global_policy().compute_dtype}')
print(f'Variable dtype: {tf.keras.mixed_precision.global_policy().variable_dtype}')

# --- 2. Verify GPU is available ---
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU detected: {gpus[0].name}')
    # Allow memory growth — prevents Colab OOM crashes
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print('WARNING: No GPU detected — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# CELL 2: Kaggle Credentials and Dataset Download
# FIX #1: kaggle reads KAGGLE_USERNAME + KAGGLE_KEY, not KAGGLE_API_TOKEN
import os, glob

os.environ['KAGGLE_USERNAME'] = 'kruinfosec'  # <-- replace
os.environ['KAGGLE_KEY']      = '0595034e9978007f27bea6c159f47f24'   # <-- replace

DATA_DIR = 'data/raw'
os.makedirs(DATA_DIR, exist_ok=True)

if not glob.glob(os.path.join(DATA_DIR, '**/*.csv'), recursive=True):
    print('Downloading BoT-IoT dataset from Kaggle...')
    !kaggle datasets download -d majedjaber/bot-iot-all-features-5-sample -p {DATA_DIR} --unzip
else:
    n = len(glob.glob(os.path.join(DATA_DIR, '**/*.csv'), recursive=True))
    print(f'Dataset already present ({n} CSV files)')

Dataset already present (4 CSV files)


In [3]:
# CELL 3: Load Data
# FIX #2: LabelEncoder from sklearn.preprocessing (was sklearn.decomposition)
# FIX #3: SAMPLE_FRAC = 0.20  (was 0.01 — too small to replicate results)
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder  # FIX #2 and #4
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, '**/*.csv'), recursive=True))
print(f'Loading {len(csv_files)} CSV file(s)...')
df = pd.concat([pd.read_csv(f, low_memory=False) for f in csv_files], ignore_index=True)
print(f'Total records: {len(df):,}  |  Columns: {len(df.columns)}')

# FIX #3: use 1.0 for full replication; 0.20 for speed patch
SAMPLE_FRAC = 0.20
if SAMPLE_FRAC < 1.0:
    df = df.sample(frac=SAMPLE_FRAC, random_state=42).reset_index(drop=True)
    print(f'Sampled to {len(df):,} records ({SAMPLE_FRAC*100:.0f}%)')

Loading 4 CSV file(s)...
Total records: 3,668,522  |  Columns: 46


In [4]:
# CELL 4: Clean and Encode
df = df.replace([np.inf, -np.inf], np.nan).drop_duplicates()
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]:
            df[col] = df[col].fillna(df[col].mean())
        else:
            df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'unknown')
print(f'After cleaning: {len(df):,} records')

target_col = None
for candidate in ['attack', 'Attack', 'label', 'Label', 'category', 'Category']:
    if candidate in df.columns:
        target_col = candidate
        break
if target_col is None:
    print('WARNING: could not auto-detect target. Available columns:'); print(df.columns.tolist())
    target_col = input('Enter target column name: ')
print(f'Target: {target_col}\n{df[target_col].value_counts()}')

exclude_cols = {target_col}
for col in df.columns:
    if col.lower() in ['pkseqid','saddr','daddr','sport','dport','category','subcategory'] and col != target_col:
        exclude_cols.add(col)

cat_cols = [c for c in df.select_dtypes(include=['object','category']).columns if c not in exclude_cols]
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype=int)
    print(f'One-hot encoded {len(cat_cols)} cols. Total features: {len(df.columns)}')

feature_cols = [c for c in df.columns if c not in exclude_cols]
X = df[feature_cols].values.astype(np.float32)
le = LabelEncoder()
y  = le.fit_transform(df[target_col])
print(f'Classes: {dict(zip(le.classes_, range(len(le.classes_))))}  |  X shape: {X.shape}')

N_CLASSES = len(np.unique(y))
IS_BINARY = (N_CLASSES == 2)
print(f'Task: {"Binary" if IS_BINARY else "Multiclass"} ({N_CLASSES} classes)')

After cleaning: 3,668,522 records
Target: attack
attack
1    3668045
0        477
Name: count, dtype: int64
One-hot encoded 3 cols. Total features: 68
Classes: {np.int64(0): 0, np.int64(1): 1}  |  X shape: (3668522, 60)
Task: Binary (2 classes)


In [5]:
# CELL 5: Split, Scale, PCA, SMOTE
# FIX #4: MinMaxScaler [0,1] per paper Algorithm 2
# FIX #5: PCA fit only on X_train to prevent data leakage

# 70/15/15 split matching paper
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val,   X_test,  y_val,  y_test  = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
print(f'Split: Train={len(X_train):,} | Val={len(X_val):,} | Test={len(X_test):,}')

# FIX #4: Min-Max scaling as per paper Algorithm 2
scaler  = MinMaxScaler(feature_range=(0, 1))
X_train = scaler.fit_transform(X_train)   # fit on train only
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# FIX #5: PCA fit on train only
pca     = PCA(n_components=0.95, random_state=42)
X_train = pca.fit_transform(X_train)      # fit on train only
X_val   = pca.transform(X_val)
X_test  = pca.transform(X_test)
print(f'PCA components retained: {X_train.shape[1]}')

print(f'Before SMOTE: {dict(pd.Series(y_train).value_counts())}')
smote_pipeline = ImbPipeline([
    ('smote',      SMOTE(random_state=42)),
    ('undersample', RandomUnderSampler(random_state=42))
])
X_train, y_train = smote_pipeline.fit_resample(X_train, y_train)
print(f'After SMOTE:  {dict(pd.Series(y_train).value_counts())}')

X_train_dl = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_val_dl   = X_val.reshape((X_val.shape[0],     X_val.shape[1],   1))
X_test_dl  = X_test.reshape((X_test.shape[0],   X_test.shape[1],  1))
print(f'DL input shape: {X_train_dl.shape}')

Split: Train=2,567,965 | Val=550,278 | Test=550,279
PCA components retained: 7
Before SMOTE: {1: np.int64(2567631), 0: np.int64(334)}
After SMOTE:  {0: np.int64(2567631), 1: np.int64(2567631)}
DL input shape: (5135262, 7, 1)


In [ ]:
# ============================================================
# REPLACEMENT for Cell 6 — all 6 model builders
# Only difference from fixed notebook: dtype='float32' on
# all output Dense layers (required for mixed precision).
# ============================================================
from tensorflow.keras import layers, models, regularizers

INPUT_SHAPE = (X_train_dl.shape[1], X_train_dl.shape[2])
N_OUT   = 1 if IS_BINARY else N_CLASSES
LOSS    = 'binary_crossentropy' if IS_BINARY else 'sparse_categorical_crossentropy'
OUT_ACT = 'sigmoid' if IS_BINARY else 'softmax'

def build_lstm_cnn(input_shape):
    inp = layers.Input(shape=input_shape)
    x   = inp
    x = layers.LSTM(256, activation='tanh', return_sequences=True,
                    kernel_regularizer=regularizers.l2(1e-4), name='lstm_1')(x)
    x = layers.Dropout(0.3, name='lstm_drop_1')(x)
    x = layers.LSTM(128, activation='tanh', return_sequences=False,
                    kernel_regularizer=regularizers.l2(1e-4), name='lstm_2')(x)
    x = layers.Dropout(0.3, name='lstm_drop_2')(x)
    x = layers.Reshape((128, 1), name='reshape_for_cnn')(x)
    for filters, tag in zip([64, 128, 256], ['1','2','3']):
        x = layers.Conv1D(filters, 3, activation='relu', padding='same', name=f'conv_{tag}')(x)
        x = layers.MaxPooling1D(pool_size=2, name=f'pool_{tag}')(x)
    x   = layers.Flatten(name='flatten')(x)
    x   = layers.Dense(128, activation='relu',
                       kernel_regularizer=regularizers.l2(1e-4), name='dense_1')(x)
    x   = layers.Dropout(0.3, name='dense_drop')(x)
    # dtype='float32' required for numerical stability with mixed precision
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32', name='output')(x)
    m   = models.Model(inp, out, name='LSTM_CNN_Hybrid')
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005, clipnorm=1.0),
              loss=LOSS, metrics=['accuracy'])
    return m

def build_cnn(input_shape):
    inp = layers.Input(shape=input_shape)
    x   = inp
    for f in [64, 128, 256]:
        x = layers.Conv1D(f, 3, activation='relu', padding='same')(x)
        x = layers.MaxPooling1D(pool_size=2)(x)
    x   = layers.Flatten()(x)
    x   = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32')(x)  # fp32 output
    m   = models.Model(inp, out, name='CNN')
    m.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss=LOSS, metrics=['accuracy'])
    return m

def build_rnn(input_shape):
    inp = layers.Input(shape=input_shape)
    x   = layers.SimpleRNN(128, return_sequences=True)(inp)
    x   = layers.Dropout(0.3)(x)
    x   = layers.SimpleRNN(128)(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32')(x)  # fp32 output
    m   = models.Model(inp, out, name='RNN')
    m.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss=LOSS, metrics=['accuracy'])
    return m

def build_lstm(input_shape):
    inp = layers.Input(shape=input_shape)
    x   = layers.LSTM(128, return_sequences=True)(inp)
    x   = layers.Dropout(0.3)(x)
    x   = layers.LSTM(128)(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32')(x)  # fp32 output
    m   = models.Model(inp, out, name='LSTM')
    m.compile(optimizer=tf.keras.optimizers.Adam(0.0005), loss=LOSS, metrics=['accuracy'])
    return m

def build_bilstm(input_shape):
    inp = layers.Input(shape=input_shape)
    x   = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(inp)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Bidirectional(layers.LSTM(128))(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32')(x)  # fp32 output
    m   = models.Model(inp, out, name='BiLSTM')
    m.compile(optimizer=tf.keras.optimizers.Adam(0.0005), loss=LOSS, metrics=['accuracy'])
    return m

def build_gru(input_shape):
    inp = layers.Input(shape=input_shape)
    x   = layers.GRU(128, return_sequences=True)(inp)
    x   = layers.Dropout(0.2)(x)
    x   = layers.GRU(128)(x)
    x   = layers.Dropout(0.2)(x)
    x   = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32')(x)  # fp32 output
    m   = models.Model(inp, out, name='GRU')
    m.compile(optimizer=tf.keras.optimizers.Adam(0.0005), loss=LOSS, metrics=['accuracy'])
    return m

ALL_MODELS = {
    'LSTM-CNN (Hybrid)': build_lstm_cnn,
    'CNN':               build_cnn,
    'RNN':               build_rnn,
    'LSTM':              build_lstm,
    'BiLSTM':            build_bilstm,
    'GRU':               build_gru,
}
print(f'Mixed precision policy: {tf.keras.mixed_precision.global_policy().name}')
print(f'Models defined: {list(ALL_MODELS.keys())}')

In [ ]:
# ============================================================
# REPLACEMENT for Cell 7 training loop
# Changes from fixed notebook:
#   EPOCHS = 30  (was 50)
#   BATCH_SIZE = 512  (was 128)
#   tf.data pipeline added for GPU prefetch (faster data loading)
#   Per-model timing printed so you can track Colab session time
# ============================================================
import time
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

EPOCHS     = 30   # early stopping typically fires at ~10–20 anyway
BATCH_SIZE = 512  # fills T4 VRAM more efficiently than 128

# tf.data pipeline: keeps GPU fed during data loading (prefetch)
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(X, y, batch_size, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(10_000, len(X)), seed=42)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_dataset(X_train_dl, y_train, BATCH_SIZE, shuffle=True)
val_ds   = make_dataset(X_val_dl,   y_val,   BATCH_SIZE)
test_ds  = make_dataset(X_test_dl,  y_test,  BATCH_SIZE)

all_results     = {}
all_histories   = {}
all_predictions = {}
trained_models  = {}

session_start = time.time()

for name, builder in ALL_MODELS.items():
    print(f'\n{"="*60}\n  Training: {name}\n{"="*60}')
    elapsed = (time.time() - session_start) / 60
    print(f'  Session time so far: {elapsed:.1f} min')

    model = builder(INPUT_SHAPE)
    model.summary()

    cbs = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=5,
            restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=3, min_lr=1e-6, verbose=1)
    ]

    start   = time.time()
    history = model.fit(
        train_ds,
        epochs=EPOCHS,
        validation_data=val_ds,
        callbacks=cbs,
        verbose=1
    )
    train_time = time.time() - start

    y_proba = model.predict(test_ds, verbose=0)

    if IS_BINARY:
        y_proba_flat = y_proba.flatten()
        y_pred       = (y_proba_flat > 0.5).astype(int)
        auc_score    = roc_auc_score(y_test, y_proba_flat)
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    else:
        y_pred       = np.argmax(y_proba, axis=1)
        y_proba_flat = y_proba.max(axis=1)
        try:
            auc_score = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
        except Exception:
            auc_score = float('nan')
        cm_full = confusion_matrix(y_test, y_pred)
        fp = (cm_full.sum(axis=0) - np.diag(cm_full)).sum()
        fn = (cm_full.sum(axis=1) - np.diag(cm_full)).sum()
        tp = np.diag(cm_full).sum()
        tn = cm_full.sum() - fp - fn - tp

    avg     = 'binary' if IS_BINARY else 'macro'
    fpr_val = round((fp / (fp + tn) * 100) if (fp + tn) > 0 else 0.0, 2)
    dr_val  = round((tp / (tp + fn) * 100) if (tp + fn) > 0 else 0.0, 2)

    metrics = {
        'Accuracy (%)':       round(accuracy_score(y_test, y_pred) * 100, 2),
        'Precision (%)':      round(precision_score(y_test, y_pred, average=avg, zero_division=0) * 100, 2),
        'Recall (%)':         round(recall_score(y_test, y_pred, average=avg, zero_division=0) * 100, 2),
        'F1-Score (%)':       round(f1_score(y_test, y_pred, average=avg, zero_division=0) * 100, 2),
        'FPR (%)':            fpr_val,
        'Detection Rate (%)': dr_val,
        'AUC-ROC (%)':        round(auc_score * 100, 2),
        'Train Time (s)':     round(train_time, 1),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
        'Epochs Run': len(history.history['loss']),
    }

    all_results[name]     = metrics
    all_histories[name]   = history.history
    all_predictions[name] = {'y_true': y_test, 'y_pred': y_pred, 'y_proba': y_proba_flat}
    trained_models[name]  = model

    print(f'\n  Accuracy:       {metrics["Accuracy (%)"]:.2f}%')
    print(f'  Precision:      {metrics["Precision (%)"]:.2f}%')
    print(f'  Recall:         {metrics["Recall (%)"]:.2f}%')
    print(f'  F1-Score:       {metrics["F1-Score (%)"]:.2f}%')
    print(f'  FPR:            {metrics["FPR (%)"]:.2f}%')
    print(f'  Detection Rate: {metrics["Detection Rate (%)"]:.2f}%')
    print(f'  AUC-ROC:        {metrics["AUC-ROC (%)"]:.2f}%')
    print(f'  Epochs run: {metrics["Epochs Run"]}  |  Time: {train_time:.0f}s')
    total_elapsed = (time.time() - session_start) / 60
    print(f'  Total session time: {total_elapsed:.1f} min')

print('\n' + '='*60 + '\n  ALL MODELS TRAINED!\n' + '='*60)
total_min = (time.time() - session_start) / 60
print(f'Total wall time: {total_min:.1f} minutes')

In [ ]:
# CELL 8: Results Comparison Table
display_cols = ['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)',
                'FPR (%)', 'Detection Rate (%)', 'AUC-ROC (%)', 'Train Time (s)', 'Epochs Run']
results_df = pd.DataFrame(all_results).T[display_cols]
results_df.index.name = 'Model'

styled = (results_df.style
          .format('{:.2f}')
          .highlight_max(axis=0,
              subset=['Accuracy (%)','Precision (%)','Recall (%)','F1-Score (%)','Detection Rate (%)','AUC-ROC (%)'],
              color='#c6efce')
          .highlight_min(axis=0, subset=['FPR (%)'], color='#c6efce')
          .set_caption('Performance Comparison — Sinha et al. (2025) Replication'))
display(styled)

In [ ]:
# CELL 9: Visualizations
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, classification_report

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0', '#00BCD4']

# 9a. LSTM-CNN training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
h = all_histories['LSTM-CNN (Hybrid)']
ax1.plot(h['loss'], label='Train', color=COLORS[0], lw=2)
ax1.plot(h['val_loss'], label='Val', color=COLORS[3], lw=2)
ax1.set_title('LSTM-CNN Hybrid — Loss', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()
ax2.plot(h['accuracy'], label='Train', color=COLORS[1], lw=2)
ax2.plot(h['val_accuracy'], label='Val', color=COLORS[4], lw=2)
ax2.set_title('LSTM-CNN Hybrid — Accuracy', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend()
plt.tight_layout(); plt.show()

# 9b. ROC Curves
if IS_BINARY:
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, (name, data) in enumerate(all_predictions.items()):
        fpr_v, tpr_v, _ = roc_curve(data['y_true'], data['y_proba'])
        ax.plot(fpr_v, tpr_v, color=COLORS[i%6], lw=2,
                label=f'{name} (AUC={auc(fpr_v,tpr_v):.4f})')
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curves — All Models', fontweight='bold')
    ax.legend(loc='lower right'); plt.tight_layout(); plt.show()

# 9c. Bar chart comparison
fig, ax = plt.subplots(figsize=(14, 6))
bar_cols = ['Accuracy (%)','Precision (%)','Recall (%)','F1-Score (%)','Detection Rate (%)']
results_df[bar_cols].plot(kind='bar', ax=ax, color=COLORS[:5], width=0.75)
ax.set_ylabel('Score (%)'); ax.set_title('Model Comparison — All Metrics', fontweight='bold')
ax.set_ylim([max(results_df[bar_cols].min().min()-2, 80), 101])
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
for c in ax.containers: ax.bar_label(c, fmt='%.2f', fontsize=6, padding=2)
ax.legend(fontsize=9, loc='lower right'); plt.tight_layout(); plt.show()

# 9d. Confusion matrices
n_models = len(all_predictions)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, (name, data) in enumerate(all_predictions.items()):
    ax = axes[idx]
    cm = confusion_matrix(data['y_true'], data['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    m = all_results[name]
    ax.set_title(f'{name}\nAcc={m["Accuracy (%)"]:.2f}%  F1={m["F1-Score (%)"]:.2f}%  FPR={m["FPR (%)"]:.2f}%',
                 fontweight='bold', fontsize=10)
for idx in range(n_models, len(axes)): axes[idx].set_visible(False)
plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# 9e. Classification reports
print('\n' + '='*60 + '\n  CLASSIFICATION REPORTS\n' + '='*60)
for name, data in all_predictions.items():
    print(f'\n--- {name} ---')
    print(classification_report(data['y_true'], data['y_pred'],
          target_names=[str(c) for c in le.classes_], digits=4))

In [ ]:
# CELL 9b: Learning Curves Comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
LINE_STYLES = ['-','--','-.', ':', '-','--']
MARKERS     = ['o','s','^', 'D','v', 'P']
for (metric_key, title, ylabel, ax) in [
    ('loss',         'Training Loss',       'Loss',     axes[0,0]),
    ('val_loss',     'Validation Loss',     'Loss',     axes[0,1]),
    ('accuracy',     'Training Accuracy',   'Accuracy', axes[1,0]),
    ('val_accuracy', 'Validation Accuracy', 'Accuracy', axes[1,1]),
]:
    for i, (name, hist) in enumerate(all_histories.items()):
        ep = range(1, len(hist[metric_key])+1)
        ax.plot(ep, hist[metric_key], color=COLORS[i%6], lw=2,
                linestyle=LINE_STYLES[i%6], marker=MARKERS[i%6],
                markersize=4, markevery=max(1, len(hist[metric_key])//10),
                label=name, alpha=0.9)
    ax.set_title(title, fontweight='bold'); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.suptitle('Learning Curves — All Models', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# CELL 10: Feature Importance (Permutation-based)
from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator, ClassifierMixin

class KerasWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, model, is_binary):
        self.model = model; self.is_binary = is_binary
        self.classes_ = np.array([0,1]) if is_binary else np.unique(y)
    def fit(self, X, y): return self
    def predict(self, X):
        p = self.model.predict(X.reshape(X.shape[0], X.shape[1], 1), verbose=0)
        return (p.flatten()>0.5).astype(int) if self.is_binary else np.argmax(p, axis=1)
    def score(self, X, y): return np.mean(self.predict(X)==y)

wrapper = KerasWrapper(trained_models['LSTM-CNN (Hybrid)'], IS_BINARY)
n_samp  = min(1000, len(X_test))
result  = permutation_importance(wrapper, X_test[:n_samp], y_test[:n_samp],
                                  n_repeats=5, random_state=42, n_jobs=-1)

sorted_idx   = result.importances_mean.argsort()[::-1][:20]
top_features = [feature_cols[i] if i < len(feature_cols) else f'PC_{i}' for i in sorted_idx]
top_imp      = result.importances_mean[sorted_idx]
top_std      = result.importances_std[sorted_idx]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(top_features)), top_imp, xerr=top_std, color=COLORS[0], alpha=0.8)
ax.set_yticks(range(len(top_features))); ax.set_yticklabels(top_features)
ax.invert_yaxis(); ax.set_xlabel('Mean Accuracy Decrease')
ax.set_title('LSTM-CNN Hybrid — Top 20 Feature Importance', fontweight='bold')
plt.tight_layout(); plt.show()

print('\nTop 10:')
for i in range(min(10, len(top_features))):
    print(f'  {i+1:2d}. {top_features[i]}: {top_imp[i]:.4f} +/- {top_std[i]:.4f}')

In [ ]:
# CELL 11: Save All Results
import json
from datetime import datetime

timestamp   = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR     = f'results/run_{timestamp}'
for d in [f'{RUN_DIR}/figures', f'{RUN_DIR}/tables', f'{RUN_DIR}/models', f'{RUN_DIR}/training_history']:
    os.makedirs(d, exist_ok=True)
print(f'Saving to: {RUN_DIR}/')

results_df.to_csv(f'{RUN_DIR}/tables/model_comparison.csv')
results_df.to_latex(f'{RUN_DIR}/tables/model_comparison.tex', float_format='%.2f')

run_meta = dict(timestamp=timestamp, sample_frac=SAMPLE_FRAC, scaler='MinMaxScaler[0,1]',
                epochs_max=EPOCHS, batch_size=BATCH_SIZE, task='binary' if IS_BINARY else 'multiclass')
full_results = {'run_config': run_meta, 'metrics': {
    n: {k: float(v) if isinstance(v,(int,float,np.integer,np.floating)) else v for k,v in m.items()}
    for n, m in all_results.items()}}
with open(f'{RUN_DIR}/tables/all_results.json','w') as f: json.dump(full_results, f, indent=2)

for name, hist in all_histories.items():
    safe = name.lower().replace(' ','_').replace('(','').replace(')','').replace('-','_')
    with open(f'{RUN_DIR}/training_history/{safe}_history.json','w') as f:
        json.dump({k:[float(v) for v in vals] for k,vals in hist.items()}, f, indent=2)

feat_df = pd.DataFrame({'Feature':top_features,'Importance':top_imp,'Std':top_std})
feat_df.to_csv(f'{RUN_DIR}/tables/feature_importance.csv', index=False)

with open(f'{RUN_DIR}/tables/classification_reports.txt','w') as f:
    for name, data in all_predictions.items():
        f.write(f'\n{"="*60}\n  {name}\n{"="*60}\n')
        f.write(classification_report(data['y_true'], data['y_pred'],
                target_names=[str(c) for c in le.classes_], digits=4))

for name, model in trained_models.items():
    safe = name.lower().replace(' ','_').replace('(','').replace(')','').replace('-','_')
    model.save(f'{RUN_DIR}/models/{safe}.keras')

print('All results saved.')
for root, dirs, files in os.walk(RUN_DIR):
    lvl = root.replace(RUN_DIR,'').count(os.sep)
    print('  '*lvl + os.path.basename(root) + '/')
    for f in sorted(files):
        kb = os.path.getsize(os.path.join(root,f))/1024
        print('  '*(lvl+1) + f + f' ({kb:.1f} KB)')

## Target Results (paper Table 6)

| Model | Accuracy | Precision | Recall | F1 | FPR |
|-------|----------|-----------|--------|----|-----|
| **LSTM-CNN (Hybrid)** | **99.87%** | **99.89%** | **99.85%** | **99.87%** | **0.13%** |
| BiLSTM | 98.92% | 98.97% | 98.87% | 98.92% | 1.03% |
| LSTM | 98.34% | 98.42% | 98.26% | 98.34% | 1.58% |
| GRU | 97.89% | 97.95% | 97.83% | 97.89% | 2.05% |
| CNN | 97.45% | 97.62% | 97.28% | 97.45% | 2.38% |
| RNN | 95.78% | 95.91% | 95.65% | 95.78% | 4.09% |

**Next steps**: Add SHAP explainability (Gap 3) and adversarial robustness testing with FGSM (Gap 2).